In [6]:
import cv2
from ultralytics import YOLO
import numpy as np

# Path to your YOLOv8 model
model_path = r"C:\Users\Kaushik\code_JN\runs\detect\train2\weights\best.pt"

# Load the YOLOv8 model
model = YOLO(model_path)

# Open the webcam
cap = cv2.VideoCapture(0)  # 0 is the default webcam

# Check if the webcam is opened correctly
if not cap.isOpened():
    print("Error: Could not access the webcam.")
    exit()

# Store previous positions and velocities
previous_positions = {}
velocity = {}

# Define very low movement threshold for high sensitivity
movement_threshold = 1  # Very low threshold to capture even small movements

# Store direction change buffer
direction_text = "Stationary"
previous_direction_text = "Stationary"

# Debounce parameters for direction
direction_debounce_counter = 0
direction_debounce_threshold = 5  # Update direction only if it persists for 5 frames

# For updating box count with delay
count_text = ""
previous_count_text = ""
frame_count_for_count_update = 10  # Increased delay for text update
count_update_counter = 0
previous_boxes_count = 0  # Track previous number of boxes

while True:
    # Read a frame from the webcam
    ret, frame = cap.read()
    if not ret:
        print("Error: Failed to capture image.")
        break

    # Run inference on the frame
    results = model(frame)

    # Get bounding boxes (x1, y1, x2, y2)
    boxes = results[0].boxes.xyxy.cpu().numpy()

    # Annotate frame with bounding boxes
    annotated_frame = results[0].plot()  # Plot annotations directly on the frame

    # Process each detected bounding box
    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = box  # Bounding box coordinates
        centroid_x = int((x1 + x2) / 2)
        centroid_y = int((y1 + y2) / 2)

        track_id = i  # Track each object by its index (can be improved later for persistent tracking)

        # Default direction is stationary
        new_direction_text = "Stationary"

        # If object was detected previously, calculate velocity and direction
        if track_id in previous_positions:
            prev_x, prev_y = previous_positions[track_id]
            dx = centroid_x - prev_x
            dy = centroid_y - prev_y

            # Calculate velocity (distance moved per frame)
            velocity[track_id] = np.sqrt(dx**2 + dy**2)

            # Detect direction based on movement
            if velocity[track_id] > movement_threshold:
                if abs(dx) > abs(dy):  # Horizontal movement is more significant
                    if dx > 0:
                        new_direction_text = "Right"
                    else:
                        new_direction_text = "Left"
                else:  # Vertical movement is more significant
                    if dy > 0:
                        new_direction_text = "Down"
                    else:
                        new_direction_text = "Up"

        # Update the object's position
        previous_positions[track_id] = (centroid_x, centroid_y)

        # Debounce the direction change
        if new_direction_text != previous_direction_text:
            direction_debounce_counter += 1
            if direction_debounce_counter >= direction_debounce_threshold:
                direction_text = new_direction_text
                previous_direction_text = new_direction_text
                direction_debounce_counter = 0  # Reset the counter after updating
        else:
            # If the direction text hasn't changed, keep the previous direction
            direction_text = previous_direction_text

        # Display the direction at the bottom of the bounding box
        cv2.putText(annotated_frame, f"{direction_text}", 
                    (int(x1), int(y2) + 20),  # Position the text below the bounding box
                    cv2.FONT_HERSHEY_SIMPLEX, 
                    0.5, (255, 255, 255), 2)

    # Update box count with delay (update every 10 frames, if box count has changed)
    if len(boxes) != previous_boxes_count:
        count_update_counter = 0  # Reset counter if box count changes
    else:
        count_update_counter += 1
    
    if count_update_counter >= frame_count_for_count_update:
        count_text = f"Boxes: {len(boxes)}"
        previous_count_text = count_text
        count_update_counter = 0  # Reset counter after updating the count
    
    # Display bounding box count
    cv2.putText(
        annotated_frame, 
        count_text, 
        (10, 30),  # Position at the top-left corner
        cv2.FONT_HERSHEY_SIMPLEX, 
        1,  # Font scale
        (0, 255, 0),  # Text color (green)
        2  # Thickness
    )

    # Show the annotated frame
    cv2.imshow("YOLOv8 Webcam", annotated_frame)

    # Exit on pressing 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

    # Update previous_boxes_count for the next frame
    previous_boxes_count = len(boxes)

# Release the webcam and close all windows
cap.release()
cv2.destroyAllWindows()



0: 480x640 3 peoples, 35.9ms
Speed: 3.0ms preprocess, 35.9ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 people, 31.5ms
Speed: 0.0ms preprocess, 31.5ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 people, 27.8ms
Speed: 0.0ms preprocess, 27.8ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 peoples, 30.7ms
Speed: 1.0ms preprocess, 30.7ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 people, 29.2ms
Speed: 0.0ms preprocess, 29.2ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 people, 27.4ms
Speed: 0.0ms preprocess, 27.4ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 people, 25.1ms
Speed: 1.0ms preprocess, 25.1ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 people, 26.8ms
Speed: 1.0ms preprocess, 26.8ms inference, 0.0ms postprocess per image at shape (1, 3, 

KeyboardInterrupt: 

In [7]:
cap.release()
cv2.destroyAllWindows()